In [3]:
import os
import cv2
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [4]:
DIRECTORY = r"data"
CATEGORIES = ["with_mask", "without_mask"]
data = []
labels = []

In [5]:
for category in CATEGORIES:
    path = os.path.join(DIRECTORY, category)

    for img in os.listdir(path):
        img_path = os.path.join(path, img)
        
        image = cv2.imread(img_path)
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (224, 224))
            image = img_to_array(image)
            image = preprocess_input(image)
            data.append(image)
            labels.append(category)
print(len(data), len(labels))          

7553 7553


In [6]:
import numpy as np
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [7]:
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

data = np.array(data,dtype="float32")
labels = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

print("Training images size:", len(X_train))
print("Testing images size:", len(X_test))


Training images size: 6042
Testing images size: 1511


In [8]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import AveragePooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Model

baseModel = MobileNetV2(weights="imagenet", include_top=False, input_tensor=Input(shape=(224, 224, 3)))
headmodel = baseModel.output
headmodel = AveragePooling2D(pool_size=(7, 7))(headmodel)
headmodel = Flatten(name="flatten")(headmodel)

headmodel = Dense(128, activation="relu")(headmodel)
headmodel = Dropout(0.5)(headmodel)
headmodel = Dense(2, activation="softmax")(headmodel)
model = Model(inputs=baseModel.input, outputs=headmodel)

for layer in baseModel.layers:
    layer.trainable = False
    

/tmp/ipykernel_20034/2626619325.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  baseModel = MobileNetV2(weights="imagenet", include_top=False, input_tensor=Input(shape=(224, 224, 3)))


In [10]:
from tensorflow.keras.optimizers import Adam

INIT_LR = 1e-4
EPOCHS = 20
BS = 32

OPTIMIZER = Adam(learning_rate=INIT_LR)
model.compile(
    loss="categorical_crossentropy",
    optimizer=OPTIMIZER,
    metrics=["accuracy"]
)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
aug = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

H = model.fit(
    aug.flow(X_train, y_train, batch_size=BS), 
    validation_data=(X_test, y_test),          
    epochs=EPOCHS                            
)

Epoch 1/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 138s 719ms/step - accuracy: 0.9174 - loss: 0.2021 - val_accuracy: 0.9914 - val_loss: 0.0337
Epoch 2/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 136s 722ms/step - accuracy: 0.9752 - loss: 0.0729 - val_accuracy: 0.9934 - val_loss: 0.0229
Epoch 3/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 136s 721ms/step - accuracy: 0.9815 - loss: 0.0550 - val_accuracy: 0.9921 - val_loss: 0.0212
Epoch 4/20
 98/189 ━━━━━━━━━━━━━━━━━━━━ 52s 574ms/step - accuracy: 0.9804 - loss: 0.0538

In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2026-06-19 23:26:00.976809: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 23:26:01.035082: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX_VNNI, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Num GPUs Available:  0
